# Python 常見效能寫法比較


- `set/dict` 查找 vs `list` 線性搜尋
- `list` vs `tuple`
- `import` 放在迴圈內外的差異
- 巢狀 `if` vs `and`
- 字串 `+` vs `join`
- `append` vs 預先配置後指定
- `append` vs `map` / list comprehension
- 分開指定 vs unpack
- 迴圈產生 list vs list comprehension
- 函數回傳 list vs generator
- filter 的幾種寫法
- `while` vs `for`

> 注意：執行時間會因為電腦、Python 版本、當下負載不同而變。

## 0. Benchmark 工具

我們先寫一個簡單的測速工具。每個方法會跑數次，取最快的一次，這樣比較不容易受到背景程式影響。

In [1]:
from time import perf_counter
import gc
import sys
import math
from itertools import islice


def benchmark(label, func, repeat=5, show_result=True):
    # Run func several times and print the best/average time.
    times = []
    result = None

    for _ in range(repeat):
        gc.collect()
        start = perf_counter()
        result = func()
        end = perf_counter()
        times.append(end - start)

    best = min(times)
    avg = sum(times) / len(times)

    if show_result:
        text = repr(result)
        if len(text) > 60:
            text = text[:57] + "..."
        print(f"{label:<35} best={best:.6f}s  avg={avg:.6f}s  result={text}")
    else:
        print(f"{label:<35} best={best:.6f}s  avg={avg:.6f}s")

    return best


def show_bench(title, items, repeat=5):
    # items: list of (label, function).
    print(f"\n=== {title} ===")
    results = []
    for label, func in items:
        t = benchmark(label, func, repeat=repeat)
        results.append((label, t))

    base = results[0][1]
    print("\n相對於第一個方法：")
    for label, t in results:
        if base > 0:
            print(f"{label:<35} {t / base:>7.2f}x")
    return results


## 1. `set/dict` 查找 vs `list` 線性搜尋

- `set` / `dict` 查找平均時間複雜度接近 **O(1)**
- `list` 查找通常是線性搜尋，時間複雜度是 **O(n)**

所以大量查找時，應該優先考慮 `set` 或 `dict`。


In [2]:
large_value = 1_000_000
search_value = large_value - 1

numbers_list = list(range(large_value))
numbers_set = set(numbers_list)
numbers_dict = {i: None for i in numbers_list}


def lookup_in_set():
    return search_value in numbers_set


def lookup_in_dict_key():
    return search_value in numbers_dict


def lookup_in_list_builtin():
    # list 的 in 還是線性搜尋，只是底層用 C 實作，比自己寫 for 快
    return search_value in numbers_list


def lookup_with_for_loop():
    for x in numbers_list:
        if x == search_value:
            return True
    return False


show_bench(
    "set/dict 查找 vs list 線性搜尋",
    [
        ("set 查找：x in set", lookup_in_set),
        ("dict key 查找：x in dict", lookup_in_dict_key),
        ("list 查找：x in list", lookup_in_list_builtin),
        ("for 迴圈線性搜尋", lookup_with_for_loop),
    ],
    repeat=3,
)


=== set/dict 查找 vs list 線性搜尋 ===
set 查找：x in set                     best=0.000002s  avg=0.000003s  result=True
dict key 查找：x in dict               best=0.000002s  avg=0.000002s  result=True
list 查找：x in list                   best=0.004974s  avg=0.006209s  result=True
for 迴圈線性搜尋                          best=0.011774s  avg=0.012162s  result=True

相對於第一個方法：
set 查找：x in set                        1.00x
dict key 查找：x in dict                  1.00x
list 查找：x in list                   2540.49x
for 迴圈線性搜尋                          6013.10x


[('set 查找：x in set', 1.9579892978072166e-06),
 ('dict key 查找：x in dict', 1.9579892978072166e-06),
 ('list 查找：x in list', 0.004974249983206391),
 ('for 迴圈線性搜尋', 0.011773584003094584)]

```python
x in my_set      # 平均 O(1)
x in my_dict     # 平均 O(1)，查 key
x in my_list     # O(n)
```

如果你會查很多次，可以先把 `list` 轉成 `set`：

```python
items = set(items)
```

## 2. `list` vs `tuple`

`list` 和 `tuple` 都可以用 index 直接取值，所以時間複雜度都是 **O(1)**。

差異通常不大。`tuple` 不可變，可能略快一點，但不要為了微小差異把程式寫得難懂。

In [3]:
repeat_count = 1_000_000

list_data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
tuple_data = (1, 2, 3, 4, 5, 6, 7, 8, 9, 10)


def access_list():
    total = 0
    data = list_data
    for _ in range(repeat_count):
        total += data[5]
    return total


def access_tuple():
    total = 0
    data = tuple_data
    for _ in range(repeat_count):
        total += data[5]
    return total


show_bench(
    "List vs Tuple：index 取值",
    [
        ("list[index]", access_list),
        ("tuple[index]", access_tuple),
    ],
    repeat=5,
)


=== List vs Tuple：index 取值 ===


list[index]                         best=0.027541s  avg=0.029328s  result=6000000


tuple[index]                        best=0.025519s  avg=0.026770s  result=6000000

相對於第一個方法：
list[index]                            1.00x
tuple[index]                           0.93x


[('list[index]', 0.02754062899998644), ('tuple[index]', 0.025518860999994786)]

- 要修改資料：用 `list`
- 不希望資料被改：用 `tuple`
- 兩者 index 取值都是 O(1)，效能差異通常不是主要問題

## 3. `import` 放在迴圈內 vs 放在外面

`import` 雖然有快取機制，不會每次都真的重新載入模組，但放在迴圈裡還是會產生額外查找成本。

一般建議：**import 放在檔案最上方**。

In [3]:
N = 200_000
nums = list(range(N))


def sqrt_import_inside_loop():
    total = 0.0
    for x in nums:
        import math
        total += math.sqrt(x)
    return round(total, 2)


# math 已經在最上面 import 過了

def sqrt_import_outside_loop():
    total = 0.0
    sqrt = math.sqrt  # 先綁定到區域變數，減少屬性查找
    for x in nums:
        total += sqrt(x)
    return round(total, 2)


show_bench(
    "import inside loop vs outside loop",
    [
        ("import 放在迴圈裡", sqrt_import_inside_loop),
        ("import 放在外面", sqrt_import_outside_loop),
    ],
    repeat=3,
)


=== import inside loop vs outside loop ===
import 放在迴圈裡                        best=0.015581s  avg=0.018511s  result=59628255.59
import 放在外面                         best=0.006657s  avg=0.007716s  result=59628255.59

相對於第一個方法：
import 放在迴圈裡                           1.00x
import 放在外面                            0.43x


[('import 放在迴圈裡', 0.015581000014208257), ('import 放在外面', 0.006657374964561313)]

```python
# 建議
import math

for x in nums:
    y = math.sqrt(x)
```

不要寫成：

```python
for x in nums:
    import math
    y = math.sqrt(x)
```

## 4. 巢狀 `if` vs `and`

下面兩段邏輯一樣：

```python
if i % 2 == 0:
    if i % 3 == 0:
        count += 1
```

```python
if i % 2 == 0 and i % 3 == 0:
    count += 1
```

`and` 有 **短路求值**：前面是 False，就不會檢查後面。

In [5]:
N = 2_000_000


def nested_if():
    count = 0
    for i in range(N):
        if i % 2 == 0:
            if i % 3 == 0:
                count += 1
    return count


def one_if_with_and():
    count = 0
    for i in range(N):
        if i % 2 == 0 and i % 3 == 0:
            count += 1
    return count


show_bench(
    "Nested if vs and",
    [
        ("巢狀 if", nested_if),
        ("if A and B", one_if_with_and),
    ],
    repeat=3,
)


=== Nested if vs and ===


巢狀 if                               best=0.086089s  avg=0.086765s  result=333334


if A and B                          best=0.082817s  avg=0.085001s  result=333334

相對於第一個方法：
巢狀 if                                  1.00x
if A and B                             0.96x


[('巢狀 if', 0.0860888379999949), ('if A and B', 0.08281693899999709)]

這兩個效能通常差不多。選擇時以 **可讀性** 為主。

如果條件很長，`and` 可能比較清楚；如果需要分層判斷，巢狀 `if` 也可以。

## 5. 字串 `+` vs `join`

大量字串合併時，不建議在迴圈裡一直：

```python
s += substring
```

比較好的方式是先把片段放進 list，最後：

```python
''.join(parts)
```

In [4]:
N = 50_000
parts = [str(i) for i in range(N)]


def string_plus_loop():
    s = ""
    for substring in parts:
        s += substring
    return len(s)


def string_join():
    s = "".join(parts)
    return len(s)


show_bench(
    "String + vs join",
    [
        ("迴圈內 s += substring", string_plus_loop),
        ("''.join(parts)", string_join),
    ],
    repeat=3,
)


=== String + vs join ===
迴圈內 s += substring                  best=0.001981s  avg=0.002242s  result=238890
''.join(parts)                      best=0.000314s  avg=0.000331s  result=238890

相對於第一個方法：
迴圈內 s += substring                     1.00x
''.join(parts)                         0.16x


[('迴圈內 s += substring', 0.001981375040486455),
 ("''.join(parts)", 0.00031404203036800027)]



大量輸出也常用同樣想法：

```python
ans = []
for x in result:
    ans.append(str(x))

sys.stdout.write('
'.join(ans))
```

## 6. `append` vs 預先配置後指定

如果你知道 list 的大小，可以先建立固定長度：

```python
arr = [None] * n
arr[i] = value
```

但在 CPython 裡，`append()` 已經很優化，所以兩者常常差異不大。

In [5]:
N = 1_000_000


def build_with_append():
    arr = []
    for x in range(N):
        arr.append(x)
    return len(arr)


def build_with_assign():
    arr = [None] * N
    for x in range(N):
        arr[x] = x
    return len(arr)


show_bench(
    "Append vs Assign",
    [
        ("arr.append(x)", build_with_append),
        ("arr = [None] * N; arr[i] = x", build_with_assign),
    ],
    repeat=3,
)


=== Append vs Assign ===
arr.append(x)                       best=0.026128s  avg=0.034812s  result=1000000
arr = [None] * N; arr[i] = x        best=0.026080s  avg=0.035809s  result=1000000

相對於第一個方法：
arr.append(x)                          1.00x
arr = [None] * N; arr[i] = x           1.00x


[('arr.append(x)', 0.026128207973670214),
 ('arr = [None] * N; arr[i] = x', 0.02608037501340732)]



- 不知道長度：用 `append()`
- 確定長度，而且要依 index 填值：可用預先配置
- 這是小優化，不如演算法複雜度重要

## 7. `append` vs `map` vs list comprehension

當你要把大量字串轉成整數時，常見寫法有三種。

In [6]:
N = 500_000
word_list = [str(i % 1000) for i in range(N)]


def int_with_append():
    new_list = []
    for word in word_list:
        new_list.append(int(word))
    return len(new_list)


def int_with_map():
    new_list = list(map(int, word_list))
    return len(new_list)


def int_with_list_comprehension():
    new_list = [int(word) for word in word_list]
    return len(new_list)


show_bench(
    "Append vs Map vs List Comprehension",
    [
        ("for + append(int(word))", int_with_append),
        ("list(map(int, word_list))", int_with_map),
        ("[int(word) for word in word_list]", int_with_list_comprehension),
    ],
    repeat=3,
)


=== Append vs Map vs List Comprehension ===
for + append(int(word))             best=0.034204s  avg=0.036996s  result=500000
list(map(int, word_list))           best=0.029183s  avg=0.030594s  result=500000
[int(word) for word in word_list]   best=0.032066s  avg=0.054647s  result=500000

相對於第一個方法：
for + append(int(word))                1.00x
list(map(int, word_list))              0.85x
[int(word) for word in word_list]      0.94x


[('for + append(int(word))', 0.034204334020614624),
 ('list(map(int, word_list))', 0.02918345801299438),
 ('[int(word) for word in word_list]', 0.03206591698108241)]

競賽輸入常用：

```python
nums = list(map(int, input().split()))
```

或大量輸入：

```python
import sys
nums = list(map(int, sys.stdin.read().split()))
```

`map(int, ...)` 對這種「套用內建函式」很常用。

## 8. 分開指定 vs unpack

兩種寫法：

```python
a = 1
b = 2
c = 3
```

```python
a, b, c = 1, 2, 3
```

這種差異非常小，實務上以可讀性為主。

In [9]:
N = 1_000_000


def separate_assign():
    a = b = c = 0
    for _ in range(N):
        a = 1
        b = 2
        c = 3
    return a + b + c


def unpack_assign():
    a = b = c = 0
    for _ in range(N):
        a, b, c = 1, 2, 3
    return a + b + c


show_bench(
    "Assign vs Unpack",
    [
        ("分開指定", separate_assign),
        ("unpack 指定", unpack_assign),
    ],
    repeat=5,
)


=== Assign vs Unpack ===


分開指定                                best=0.018752s  avg=0.019201s  result=6


unpack 指定                           best=0.016593s  avg=0.017259s  result=6

相對於第一個方法：
分開指定                                   1.00x
unpack 指定                              0.88x


[('分開指定', 0.01875183800001423), ('unpack 指定', 0.01659337600000299)]

這種不用特別優化，寫清楚比較重要。

例如交換兩個變數時，Python 很常寫：

```python
a, b = b, a
```

## 9. 迴圈 `append` vs list comprehension

建立新 list 時，list comprehension 通常比較簡潔，也常常比較快。

In [7]:
N = 1_000_000


def square_even_with_loop():
    new_list = []
    for i in range(N):
        if i % 2 == 0:
            new_list.append(i ** 2)
    return len(new_list)


def square_even_with_comprehension():
    new_list = [i ** 2 for i in range(N) if i % 2 == 0]
    return len(new_list)


show_bench(
    "迴圈 append vs list comprehension",
    [
        ("for + if + append", square_even_with_loop),
        ("list comprehension", square_even_with_comprehension),
    ],
    repeat=3,
)


=== 迴圈 append vs list comprehension ===
for + if + append                   best=0.046378s  avg=0.051252s  result=500000
list comprehension                  best=0.042040s  avg=0.044180s  result=500000

相對於第一個方法：
for + if + append                      1.00x
list comprehension                     0.91x


[('for + if + append', 0.04637783399084583),
 ('list comprehension', 0.042039541993290186)]



```python
# 原本
result = []
for i in range(n):
    if i % 2 == 0:
        result.append(i ** 2)

# 可以改成
result = [i ** 2 for i in range(n) if i % 2 == 0]
```

但如果邏輯太複雜，還是用普通 for 迴圈比較容易讀。

## 10. 函數回傳 list vs generator

Generator 的重點不一定是「永遠比較快」，而是：

- 不用一次把所有資料存在記憶體
- 可以邊產生邊使用
- 如果只需要前幾個結果，generator 非常適合

In [8]:
N = 500_000


def make_number_list(n):
    result = []
    for i in range(n):
        result.append(i * 2)
    return result


def make_number_generator(n):
    for i in range(n):
        yield i * 2


def consume_list_all():
    total = 0
    for number in make_number_list(N):
        total += number
    return total


def consume_generator_all():
    total = 0
    for number in make_number_generator(N):
        total += number
    return total


show_bench(
    "Function 回傳 list vs Generator：全部走訪",
    [
        ("先建立 list 再走訪", consume_list_all),
        ("generator 邊產生邊走訪", consume_generator_all),
    ],
    repeat=3,
)


=== Function 回傳 list vs Generator：全部走訪 ===
先建立 list 再走訪                        best=0.030761s  avg=0.068052s  result=249999500000
generator 邊產生邊走訪                    best=0.026372s  avg=0.027834s  result=249999500000

相對於第一個方法：
先建立 list 再走訪                           1.00x
generator 邊產生邊走訪                       0.86x


[('先建立 list 再走訪', 0.030760916008148342),
 ('generator 邊產生邊走訪', 0.026372166990768164)]

In [9]:
# 如果只需要前 5 個資料，generator 不需要建立完整 list

def first_5_from_list():
    return make_number_list(N)[:5]


def first_5_from_generator():
    return list(islice(make_number_generator(N), 5))


show_bench(
    "只需要前 5 個結果",
    [
        ("先建立完整 list 再切前 5 個", first_5_from_list),
        ("generator 只產生前 5 個", first_5_from_generator),
    ],
    repeat=5,
)


=== 只需要前 5 個結果 ===
先建立完整 list 再切前 5 個                  best=0.018510s  avg=0.022964s  result=[0, 2, 4, 6, 8]
generator 只產生前 5 個                  best=0.000012s  avg=0.000195s  result=[0, 2, 4, 6, 8]

相對於第一個方法：
先建立完整 list 再切前 5 個                     1.00x
generator 只產生前 5 個                     0.00x


[('先建立完整 list 再切前 5 個', 0.01851033401908353),
 ('generator 只產生前 5 個', 1.2167030945420265e-05)]

### 補充：觀察記憶體

下面用 `tracemalloc` 看 peak memory。這格可能會比前面慢一點。

In [10]:
import tracemalloc


def measure_time_and_memory(label, func):
    gc.collect()
    tracemalloc.start()
    start = perf_counter()
    result = func()
    end = perf_counter()
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    print(f"{label:<30} time={end-start:.6f}s  peak={peak/1024/1024:.2f} MB  result={result}")


measure_time_and_memory("建立 list 並 sum", consume_list_all)
measure_time_and_memory("generator 並 sum", consume_generator_all)

建立 list 並 sum                  time=0.430334s  peak=19.23 MB  result=249999500000
generator 並 sum                time=0.443458s  peak=0.00 MB  result=249999500000


## 11. Filter Method


- `for` + `append`
- list comprehension
- generator expression
- `filter()`
- 直接用數學規律產生答案

注意：generator expression 和 `filter()` 都是 lazy evaluation。只建立 iterator 不代表真的處理完資料。

In [14]:
# generator expression 不會馬上產生所有資料

g = (i for i in range(10) if i % 7 == 0)
print(g)
print(list(g))

<generator object <genexpr> at 0x7ea71c492b50>
[0, 7]


In [15]:
N = 1_000_000


def filter_with_loop():
    result = []
    for i in range(N):
        if i % 7 == 0:
            result.append(i)
    return len(result)


def filter_with_list_comprehension():
    result = [i for i in range(N) if i % 7 == 0]
    return len(result)


def filter_with_filter_function():
    result = list(filter(lambda x: x % 7 == 0, range(N)))
    return len(result)


def filter_with_generator_consumed():
    # fair：要真的消耗 generator，才算完成運算
    return sum(1 for i in range(N) if i % 7 == 0)


def filter_by_math_pattern():
    # 因為我們知道 7 的倍數是 0, 7, 14, ...，可以直接產生
    result = list(range(0, N, 7))
    return len(result)


show_bench(
    "Filter Method",
    [
        ("for + append", filter_with_loop),
        ("list comprehension", filter_with_list_comprehension),
        ("list(filter(lambda...))", filter_with_filter_function),
        ("generator consumed by sum", filter_with_generator_consumed),
        ("直接 range(0, N, 7)", filter_by_math_pattern),
    ],
    repeat=3,
)


=== Filter Method ===


for + append                        best=0.033878s  avg=0.034427s  result=142858


list comprehension                  best=0.032453s  avg=0.033222s  result=142858


list(filter(lambda...))             best=0.040742s  avg=0.041858s  result=142858


generator consumed by sum           best=0.029878s  avg=0.030148s  result=142858
直接 range(0, N, 7)                   best=0.003877s  avg=0.004160s  result=142858

相對於第一個方法：
for + append                           1.00x
list comprehension                     0.96x
list(filter(lambda...))                1.20x
generator consumed by sum              0.88x
直接 range(0, N, 7)                      0.11x


[('for + append', 0.033877944000010984),
 ('list comprehension', 0.03245302999999922),
 ('list(filter(lambda...))', 0.04074206099997468),
 ('generator consumed by sum', 0.02987812800000711),
 ('直接 range(0, N, 7)', 0.0038765139999838993)]



- 一般篩選：list comprehension 很常用
- 只要逐一使用，不需要存成 list：generator 可以省記憶體
- 如果題目有數學規律，直接利用規律通常最快

## 12. `while` vs `for`

很多時候：

```python
for i in range(n):
```

會比手動控制 `while` 更簡潔，也常常更快。

In [11]:
N = 2_000_000


def sum_with_while():
    total = 0
    i = 0
    while i < N:
        total += i
        i += 1
    return total


def sum_with_for():
    total = 0
    for i in range(N):
        total += i
    return total


def sum_with_builtin():
    return sum(range(N))


show_bench(
    "While vs For vs Built-in sum",
    [
        ("while 手動控制 i", sum_with_while),
        ("for i in range(N)", sum_with_for),
        ("sum(range(N))", sum_with_builtin),
    ],
    repeat=3,
)


=== While vs For vs Built-in sum ===
while 手動控制 i                        best=0.078595s  avg=0.081555s  result=1999999000000
for i in range(N)                   best=0.054619s  avg=0.055132s  result=1999999000000
sum(range(N))                       best=0.023364s  avg=0.023435s  result=1999999000000

相對於第一個方法：
while 手動控制 i                           1.00x
for i in range(N)                      0.69x
sum(range(N))                          0.30x


[('while 手動控制 i', 0.07859520800411701),
 ('for i in range(N)', 0.054619083006400615),
 ('sum(range(N))', 0.023363540996797383)]



- 固定次數迴圈：優先用 `for i in range(n)`
- 需要「直到某條件成立才停」：用 `while`
- 能用內建函式時，通常內建函式更快也更清楚，例如 `sum()`、`max()`、`min()`

# 總結：

| 情境 | 建議寫法 | 原因 |
|---|---|---|
| 大量查找是否存在 | `set` / `dict` | 平均 O(1) |
| 大量輸入 | `sys.stdin.read()` | 比一行一行 `input()` 快 |
| 大量輸出 | `"\n".join(...)` + `sys.stdout.write()` | 減少多次輸出成本 |
| 大量字串合併 | `"".join(parts)` | 避免反覆建立新字串 |
| 建立 list | list comprehension | 簡潔，通常較快 |
| 轉型，例如 str -> int | `map(int, data)` 或 comprehension | 常用且快 |
| 不需要一次存全部 | generator | 省記憶體，可 lazy evaluation |
| 固定次數迴圈 | `for i in range(n)` | 比 `while` 清楚 |
| 有內建函式可用 | `sum/max/min/sorted` | 通常快且可讀 |

最重要：**先選對演算法，再做語法層級的小優化。**

# 課堂練習

## 練習 1：查詢黑名單

有 100 萬個黑名單 ID，接著有 10 萬次查詢。請問應該用 `list` 還是 `set` 儲存黑名單？為什麼？

## 練習 2：輸出很多答案

如果有 100 萬筆答案要輸出，一行一個答案，請寫出比較快的輸出方式。

## 練習 3：找 7 的倍數

請比較下面三種方法：

1. for + append
2. list comprehension
3. `range(0, n, 7)`

哪一個最快？為什麼？

# 參考答案

## 練習 1

應該用 `set`。因為查詢是否存在時，`x in set` 平均接近 O(1)，`x in list` 是 O(n)。

## 練習 2

```python
import sys

ans = []
for x in result:
    ans.append(str(x))

sys.stdout.write("\n".join(ans))
```

## 練習 3

如果題目就是要找 7 的倍數，`range(0, n, 7)` 通常最快，因為它直接利用數學規律，不需要每個數都做 `% 7` 判斷。